In [1]:
import sys, re, json
from pathlib import Path
from difflib import SequenceMatcher
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.common.config import PROCESSED, GOLD, TEMPLATES, ask_json, embed
import pandas as pd
import numpy as np

SCHEMA = json.loads((TEMPLATES / "test_plan_schema.json").read_text(encoding="utf-8"))
SECTIONS = {s["key"]: s["description"] for s in SCHEMA["sections"]}
REQUIRED = [s["key"] for s in SCHEMA["sections"] if s["required"]]

def project_dir(prefix, root=PROCESSED):
    return next(p for p in root.glob("*/*") if p.name.startswith(prefix))

def load_inputs(prefix):
    """Everything the evaluator needs for one project."""
    p, g = project_dir(prefix), project_dir(prefix, GOLD)
    return {"name": p.name,
            "brd": (p / "brd.md").read_text(encoding="utf-8"),
            "arch": (p / "architecture.md").read_text(encoding="utf-8"),
            "gold": json.loads((g / "gold_testplan.json").read_text(encoding="utf-8")),
            "facts": json.loads((g / "facts.json").read_text(encoding="utf-8"))["facts"]}

ctx = load_inputs("P06")
print(ctx["name"], "|", len(ctx["facts"]), "facts |", sum(v["present"] for v in ctx["gold"]["sections"].values()), "gold sections")

P06_Rebate_Validation | 98 facts | 23 gold sections


In [2]:
def _norm(s):
    s = s.lower().replace("–", "-").replace("—", "-").replace("’", "'").replace("“", '"').replace("”", '"')
    s = re.sub(r"[|*#`]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def quote_score(quote, source):
    q, src = _norm(quote), _norm(source)
    if not q:
        return 0.0
    if q in src:
        return 1.0
    m = SequenceMatcher(None, src, q, autojunk=False).find_longest_match(0, len(src), 0, len(q))
    return m.size / len(q)

def md_sections(md):
    """Split a generated markdown plan on #-headings -> [{heading, level, text}]."""
    secs, cur = [], {"heading": "(front matter)", "level": 0, "text": []}
    for line in md.splitlines():
        m = re.match(r"^(#{1,4})\s+(.*)", line)
        if m:
            if cur["text"] or cur["level"] > 0:
                secs.append({**cur, "text": "\n".join(cur["text"]).strip()})
            cur = {"heading": m.group(2).strip(), "level": len(m.group(1)), "text": []}
        else:
            cur["text"].append(line)
    if cur["text"] or cur["level"] > 0:
        secs.append({**cur, "text": "\n".join(cur["text"]).strip()})
    return secs

MAP_SYSTEM = ("You are a senior QA lead. You map headings of a software test plan to a fixed list of "
              "canonical section keys. Judge by the heading AND its content preview, not the heading alone. "
              "A sub-section usually belongs to the same key as its parent unless its content clearly fits another key "
              "(e.g. 'Security' under 'Non-Functional Testing' -> security_compliance). A merged section such as "
              "'In Scope / Out of Scope' gets BOTH keys (max 2). Use 'other' only if nothing fits.")
MAP_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["mapping"],
    "properties": {"mapping": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["index", "keys"],
        "properties": {"index": {"type": "integer"},
                       "keys": {"type": "array", "items": {"type": "string", "enum": list(SECTIONS) + ["other"]}}}}}}}

def to_canonical(secs):
    """Group plan sections into {canonical_key: text} using the same LLM mapping as the gold data."""
    keys_txt = "\n".join(f"- {k}: {d}" for k, d in SECTIONS.items())
    rows, parent = [], ""
    for i, s in enumerate(secs):
        if s["level"] == 1:
            parent = s["heading"]
        preview = re.sub(r"\s+", " ", s["text"])[:220]          # computed outside the f-string
        par = parent if s["level"] > 1 else "-"
        rows.append(f"[{i}] H{s['level']} {s['heading']} | parent: {par} | {preview}")
    out = ask_json(MAP_SYSTEM, f"CANONICAL KEYS:\n{keys_txt}\n\nTEST PLAN SECTIONS:\n" + "\n".join(rows) +
                   "\n\nReturn one mapping entry for EVERY section index.", MAP_SCHEMA, "section_mapping")
    m = {e["index"]: e["keys"][:2] for e in out["mapping"]}
    grouped = {}
    for i, s in enumerate(secs):
        ks = ["document_control"] if s["level"] == 0 else (m.get(i) or ["other"])
        for k in ks:
            if k != "other":
                block = (f"### {s['heading']}\n" if s["level"] > 1 else "") + s["text"]
                grouped[k] = (grouped.get(k, "") + "\n\n" + block).strip()
    return grouped

In [3]:
def score_structure(grouped, min_words=15):
    present = {k for k, t in grouped.items() if len(t.split()) >= min_words}
    missing = [k for k in REQUIRED if k not in present]
    return {"structure": 1 - len(missing) / len(REQUIRED), "missing_sections": missing}

In [4]:
COV_SYSTEM = ("You are a QA auditor. For each requirement fact, decide whether the TEST PLAN covers it. "
              "'yes' = the plan states or plans to verify this fact with its key specifics (value, system, role, rule); "
              "wording may differ. 'partial' = the topic is mentioned but the key specific is missing or vague. "
              "'no' = absent. For yes/partial, plan_quote = the SHORTEST exact span copied from the test plan "
              "(max 20 words) that shows it; empty for 'no'.")
COV_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["results"],
    "properties": {"results": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["id", "verdict", "plan_quote"],
        "properties": {"id": {"type": "string"},
                       "verdict": {"type": "string", "enum": ["yes", "partial", "no"]},
                       "plan_quote": {"type": "string"}}}}}}

def score_coverage(plan_md, facts, chunk_size=25, quote_min=0.6):
    res = {}
    for i in range(0, len(facts), chunk_size):                       # small batches = steadier judge
        chunk = facts[i:i + chunk_size]
        listing = "\n".join(f"{f['id']} [{f['category']}|{f['mvp']}] {f['fact']}" for f in chunk)
        out = ask_json(COV_SYSTEM, f"<test_plan>\n{plan_md}\n</test_plan>\n\n<facts>\n{listing}\n</facts>\n\n"
                       "Return one result for EVERY fact id.", COV_SCHEMA, "fact_coverage")
        res.update({r["id"]: r for r in out["results"]})
    rows = []
    for f in facts:
        r = res.get(f["id"], {"verdict": "no", "plan_quote": ""})
        judge = r["verdict"]
        qs = quote_score(r["plan_quote"], plan_md) if judge != "no" else 0.0
        final = judge if (judge == "no" or qs >= quote_min) else "no"   # evidence must exist in the plan
        rows.append({**f, "judge": judge, "verdict": final, "quote_ok": round(qs, 2), "plan_quote": r["plan_quote"],
                     "points": {"yes": 1.0, "partial": 0.5, "no": 0.0}[final]})
    df = pd.DataFrame(rows)
    must = df[df.must_test]
    return {"coverage": float(must.points.mean()) if len(must) else 0.0,
            "coverage_all": float(df.points.mean()),
            "guard_downgrades": int(((df.judge != "no") & (df.verdict == "no")).sum()),
            "missed_facts": must[must.verdict == "no"][["id", "category", "fact"]].to_dict("records"),
            "coverage_by_category": df.groupby("category").points.mean().round(2).to_dict()}, df

In [5]:
FAITH_SYSTEM = (
    "You are a QA auditor checking a generated test plan for invented content.\n"
    "Extract every PROJECT-SPECIFIC claim as an ATOMIC statement: exactly one value, system, role, rule or scope item "
    "per claim (split compound sentences). Ignore generic testing practice ('defects are logged', 'regression testing').\n"
    "kind: 'requirement' = asserts something about the system or project (scope, MVP assignment, features, rules, "
    "NFR targets, systems, integrations, data sources, roles). 'test_design' = a choice made by the test team "
    "(test data values, worked examples, coverage targets, test tools, techniques, test schedule).\n"
    "verdict vs SOURCE: 'supported' = stated in source; 'derived' = correctly follows from or is calculated from "
    "source (e.g. an arithmetic example consistent with stated tiers); 'contradicted' = source says otherwise; "
    "'not_found' = not in source.\n"
    "source_quote = SHORTEST exact span copied from the source (max 20 words) backing the verdict; empty if not_found.")
FAITH_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["claims"],
    "properties": {"claims": {"type": "array", "items": {
        "type": "object", "additionalProperties": False,
        "required": ["claim", "kind", "verdict", "source_quote"],
        "properties": {"claim": {"type": "string"},
                       "kind": {"type": "string", "enum": ["requirement", "test_design"]},
                       "verdict": {"type": "string", "enum": ["supported", "derived", "contradicted", "not_found"]},
                       "source_quote": {"type": "string"}}}}}}

NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?\s*%?")
def numbers_in(text):
    return {n.replace(",", "").replace(" ", "") for n in NUM_RE.findall(text)}

def backed(c, source, quote_min=0.6):
    """A supported/derived verdict must be backed by a real source quote OR all its numbers must exist in source."""
    if quote_score(c["source_quote"], source) >= quote_min:
        return True
    nums = numbers_in(c["claim"])
    return c["verdict"] == "supported" and bool(nums) and nums <= numbers_in(source)

def score_faithfulness(plan_md, brd, arch):
    source = brd + "\n" + arch
    out = ask_json(FAITH_SYSTEM, f"<source>\n<brd>\n{brd}\n</brd>\n<architecture>\n{arch}\n</architecture>\n</source>\n\n"
                   f"<test_plan>\n{plan_md}\n</test_plan>", FAITH_SCHEMA, "faithfulness", effort="medium")
    claims = out["claims"]
    for c in claims:
        c["ok"] = c["verdict"] in ("supported", "derived") and backed(c, source)
    req = [c for c in claims if c["kind"] == "requirement"]
    bad = [c for c in req if not c["ok"]] + \
          [c for c in claims if c["kind"] == "test_design" and c["verdict"] == "contradicted"]
    return {"faithfulness": sum(c["ok"] for c in req) / len(req) if req else 1.0,
            "claims": len(claims), "requirement_claims": len(req),
            "test_design_claims": len(claims) - len(req), "invented": len(bad),
            "invented_claims": [{"claim": c["claim"], "kind": c["kind"], "verdict": c["verdict"]} for c in bad]}

In [6]:
SIM_SYSTEM = ("You compare a GENERATED test plan with the REFERENCE test plan written by a QA lead, section by section. "
              "For each reference section key, score 0-5 how well the generated plan covers the same testing points "
              "(anywhere in the generated plan): 5 = all points, 3 = about half, 0 = none. Ignore wording and order. "
              "List the most important missing points briefly.")
SIM_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["sections"],
    "properties": {"sections": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["key", "score", "missing"],
        "properties": {"key": {"type": "string"}, "score": {"type": "integer"},
                       "missing": {"type": "array", "items": {"type": "string"}}}}}}}

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def score_similarity(grouped, gold):
    ref = {k: v["text"] for k, v in gold["sections"].items() if v["present"] and k != "document_control"}
    ref_txt = "\n\n".join(f"## [{k}]\n{t}" for k, t in ref.items())
    gen_txt = "\n\n".join(f"## [{k}]\n{t}" for k, t in grouped.items())
    out = ask_json(SIM_SYSTEM, f"<reference>\n{ref_txt}\n</reference>\n\n<generated>\n{gen_txt}\n</generated>\n\n"
                   "Return one entry for EVERY reference key.", SIM_SCHEMA, "similarity", effort="medium")
    llm_scores = {s["key"]: max(0, min(5, s["score"])) / 5 for s in out["sections"] if s["key"] in ref}
    # embeddings: same canonical section, reference vs generated (missing section = 0)
    both = [k for k in ref if grouped.get(k)]
    emb = {}
    if both:
        vecs = embed([ref[k][:6000] for k in both] + [grouped[k][:6000] for k in both])
        emb = {k: cosine(vecs[i], vecs[i + len(both)]) for i, k in enumerate(both)}
    return {"similarity": float(np.mean([llm_scores.get(k, 0) for k in ref])),
            "embedding_sim": float(np.mean([emb.get(k, 0) for k in ref])),
            "section_scores": {k: round(llm_scores.get(k, 0), 2) for k in ref},
            "missing_points": {s["key"]: s["missing"] for s in out["sections"] if s["missing"]}}

In [7]:
WEIGHTS = {"structure": 0.15, "coverage": 0.35, "faithfulness": 0.30, "similarity": 0.20}

def score_plan(plan_md, prefix, sections=None):
    """plan_md: generated plan as markdown. sections: optional pre-parsed [{heading, level, text}]."""
    ctx = load_inputs(prefix)
    grouped = to_canonical(sections or md_sections(plan_md))
    s = score_structure(grouped)
    c, cov_df = score_coverage(plan_md, ctx["facts"])
    f = score_faithfulness(plan_md, ctx["brd"], ctx["arch"])
    m = score_similarity(grouped, ctx["gold"])
    result = {"project": ctx["name"], **s, **c, **f, **m}
    result["overall"] = sum(result[k] * w for k, w in WEIGHTS.items())
    return result, cov_df

def summary(r):
    keys = ["overall", "structure", "coverage", "coverage_all", "faithfulness", "invented", "similarity", "embedding_sim"]
    return {k: (round(r[k] * 100, 1) if isinstance(r[k], float) else r[k]) for k in keys}

In [8]:
# CELL 8 – the reference plan must score HIGH
ref_md = (project_dir("P06") / "testplan.md").read_text(encoding="utf-8")
ref_secs = json.loads((project_dir("P06") / "testplan_sections.json").read_text(encoding="utf-8"))
r_ref, cov_ref = score_plan(ref_md, "P06", sections=ref_secs)
summary(r_ref)

{'overall': 90.8,
 'structure': 100.0,
 'coverage': 73.8,
 'coverage_all': 71.4,
 'faithfulness': 100.0,
 'invented': 0,
 'similarity': 100.0,
 'embedding_sim': 99.3}

In [9]:
print("guard downgrades:", r_ref["guard_downgrades"],
      "| requirement claims:", r_ref["requirement_claims"], "| test-design claims:", r_ref["test_design_claims"])
print("\nInvented (should be ~0 for the reference):")
for c in r_ref["invented_claims"]: print(" -", c)
print("\nMust-test facts the human reference plan did NOT cover:")
pd.DataFrame(r_ref["missed_facts"])

guard downgrades: 3 | requirement claims: 85 | test-design claims: 21

Invented (should be ~0 for the reference):

Must-test facts the human reference plan did NOT cover:


,id,category,fact
0,F013,nfr_performance,Cut claim preparation time from 3 days to 4 ho...
1,F040,nfr_availability,Availability requirement is 99% business hours
2,F046,data,Rebate agreements data owner is Supplier Programs
3,F047,data,Rebate agreements security classification is R...
4,F049,data,Purchase data owner is Finance Data and securi...
5,F052,data,Calculations data source is Tool output and fr...
6,F053,data,Calculations owner is Supplier Programs and se...
7,F067,assumption,Agreements are mostly digital PDFs
8,F079,functional,Analyst uploads agreement and terms are extrac...
9,F091,integration,App Gateway is entry point for Rebate Analysts...


In [10]:
# CELL 9 – a weak generic plan must score LOW
weak_md = """# Test Plan
## Objectives
Verify that the application works as expected and meets business requirements.
## Scope
All features of the application will be tested.
## Approach
Functional, regression and performance testing will be performed by the QA team.
## Defects
Defects will be logged in the tracking tool and fixed by developers.
"""
r_weak, _ = score_plan(weak_md, "P06")
summary(r_weak)

{'overall': 34.2,
 'structure': 0.0,
 'coverage': 9.4,
 'coverage_all': 7.7,
 'faithfulness': 100.0,
 'invented': 1,
 'similarity': 4.5,
 'embedding_sim': 9.8}

In [11]:
# CELL 10 – inject false facts: faithfulness must DROP
bad_md = (ref_md.replace("99%", "99.99%").replace("Databricks", "Snowflake")
                .replace("Azure AD", "Okta") + "\n\n## Extra\nLoad test with 10,000 concurrent users on Oracle EBS.")
r_bad, _ = score_plan(bad_md, "P06")
print(summary(r_bad))
r_bad["invented_claims"]

{'overall': 84.4, 'structure': 100.0, 'coverage': 65.6, 'coverage_all': 65.3, 'faithfulness': 90.6, 'invented': 8, 'similarity': 96.4, 'embedding_sim': 96.3}


[{'claim': 'Product line mapping frequency is monthly',
  'kind': 'requirement',
  'verdict': 'supported'},
 {'claim': 'Product line mapping owner is Program Admin',
  'kind': 'requirement',
  'verdict': 'supported'},
 {'claim': 'Edits to extracted terms are recorded with user and time',
  'kind': 'requirement',
  'verdict': 'not_found'},
 {'claim': 'Test plan in-scope functional item: Snowflake SQL retrieval',
  'kind': 'test_design',
  'verdict': 'contradicted'},
 {'claim': 'Test plan includes Okta as an in-scope identity provider',
  'kind': 'test_design',
  'verdict': 'contradicted'},
 {'claim': 'Integration tests will exercise calculation service against Snowflake SQL (Delta Lake)',
  'kind': 'test_design',
  'verdict': 'contradicted'},
 {'claim': 'Data retrieval tests will use Snowflake query by supplier, period and product lines with timeout and retry',
  'kind': 'test_design',
  'verdict': 'contradicted'},
 {'claim': 'Test plan specifies Okta login, MFA, and JWT validation as s

In [12]:
# CELL 11 – side by side
pd.DataFrame({"reference": summary(r_ref), "weak": summary(r_weak), "injected": summary(r_bad)})

,reference,weak,injected
overall,90.8,34.2,84.4
structure,100.0,0.0,100.0
coverage,73.8,9.4,65.6
coverage_all,71.4,7.7,65.3
faithfulness,100.0,100.0,90.6
invented,0.0,1.0,8.0
similarity,100.0,4.5,96.4
embedding_sim,99.3,9.8,96.3
